# Crop Recommendation Model Training

This notebook trains a Random Forest model to predict the best crop based on soil and weather conditions.
It uses the standard Crop Recommendation dataset.

In [ ]:
!pip install pandas numpy scikit-learn joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import joblib
import requests
import io

## 1. Load Dataset

In [ ]:
# URL to the raw CSV file
url = "https://raw.githubusercontent.com/danielchristopher513/Crop_Recommendation_Using_Machine_Learning/main/Crop_recommendation.csv"

try:
    response = requests.get(url)
    response.raise_for_status()
    df = pd.read_csv(io.StringIO(response.text))
    print("Dataset loaded successfully!")
    print(df.head())
except Exception as e:
    print(f"Error loading dataset: {e}")
    # Fallback: Try another URL if the first one fails
    url_backup = "https://raw.githubusercontent.com/amanattar/crop/main/crop_recommendation.csv"
    response = requests.get(url_backup)
    df = pd.read_csv(io.StringIO(response.text))
    print("Dataset loaded from backup!")

## 2. Preprocessing

In [ ]:
# Check for missing values
print(df.isnull().sum())

# Separate features and target
X = df[['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']]
y = df['label']

# Encode target labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Save the label encoder classes for later mapping
print("Classes:", le.classes_)

## 3. Train Model

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# Train Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# Evaluate
y_pred = rf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

## 4. Export Model

In [ ]:
# Save model and label encoder
joblib.dump(rf, 'crop_model.pkl')
joblib.dump(le, 'label_encoder.pkl')

print("Files saved: crop_model.pkl, label_encoder.pkl")
print("Please download these files and place them in the 'ml' directory of your project.")

In [ ]:
# Test prediction
test_input = np.array([[90, 42, 43, 20.8, 82.0, 6.5, 202.9]]) # Rice conditions
prediction = rf.predict(test_input)
predicted_crop = le.inverse_transform(prediction)
print(f"Test Prediction: {predicted_crop[0]}")